# YOLOv12 sign condition detection

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from ultralytics import YOLO

# Load YOLOv12 model
model = YOLO("yolo12n.pt")  # n = nano (fast). Try s/m if GPU allows


## Training, and Plot of Training Loss over Epochs

In [3]:
# Train
model.train(
    data="C:\\ARI3129_work\\Datasets\\YOLO_COCO_condition\\data.yaml",  # <-- path to your data.yaml
    epochs=50,
    imgsz=640,
    batch=8,
    device="cpu",
    workers=0,    
    max_det=100,
)

Ultralytics 8.4.5  Python-3.10.19 torch-2.9.1+cpu CPU (Intel Core(TM) i7-8700 3.20GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\ARI3129_work\Datasets\YOLO_COCO_condition\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=100, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002B062F05D80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [6]:
# Point this to your training run folder
RUN_DIR = Path("runs/detect/train4")  # change to train, train3, etc.
csv_path = RUN_DIR / "results.csv"

df = pd.read_csv(csv_path)

# Ultralytics column names can vary slightly by version; these are common:
# train/box_loss, train/cls_loss, train/dfl_loss
box_col = "train/box_loss"
cls_col = "train/cls_loss"
dfl_col = "train/dfl_loss"

# show available columns if something doesn't match
missing = [c for c in [box_col, cls_col, dfl_col] if c not in df.columns]
if missing:
    print("Missing columns:", missing)
    print("Available columns:", list(df.columns))
    raise KeyError("Your results.csv column names differ. Pick the right ones from the list above.")

epochs = df["epoch"] if "epoch" in df.columns else range(1, len(df) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, df[box_col], label="Box Loss")
plt.plot(epochs, df[cls_col], label="Class Loss")
plt.plot(epochs, df[dfl_col], label="DFL Loss")
plt.title("YOLO Training Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

out_path = RUN_DIR / "training_loss_plot.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved:", out_path)


<Figure size 1000x500 with 1 Axes>

Saved: runs\detect\train4\training_loss_plot.png


## Validation

In [8]:
model = YOLO("runs/detect/train4/weights/best.pt")
model.val(data="C:\\ARI3129_work\\Datasets\\YOLO_COCO_condition\\data.yaml")

Ultralytics 8.4.5  Python-3.10.19 torch-2.9.1+cpu CPU (Intel Core(TM) i7-8700 3.20GHz)
YOLOv12n summary (fused): 159 layers, 2,557,313 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1406.3152.6 MB/s, size: 3989.4 KB)
val: Scanning C:\ARI3129_work\Datasets\YOLO_COCO_condition\labels\val.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
val: C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\val\0396b469-IMG_20260108_180701.jpg: corrupt JPEG restored and saved
val: C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\val\043a38a9-IMG20260105131300.jpg: corrupt JPEG restored and saved
val: C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\val\0c61b297-IMG20260105131926.jpg: corrupt JPEG restored and saved
val: C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\val\25ff97a8-IMG20260105132835.jpg: corrupt JPEG restored and saved
val: C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\val\2934d3dc-IMG20260105131845.jpg: co

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002B0076AFFD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

## Testing

In [9]:
# Get Images with boxes
results = model.predict(
    source="C:\\ARI3129_work\\Datasets\\YOLO_COCO_condition\\images\\test",   # folder or image
    imgsz=960,
    conf=0.25,
    save=True
)

# Get metrics of test split
metrics = model.val(
    data=r"C:\\ARI3129_work\\Datasets\\YOLO_COCO_condition\\data.yaml",
    split="test",
    imgsz=960,
    conf=0.25,
    save=False,     
    plots=False      
)

image 1/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\028195a4-IMG20251123165145.jpg: 960x736 1 Good, 445.9ms
image 2/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\02995950-IMG20260105125533.jpg: 960x736 1 Good, 461.1ms
image 3/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\02fdc633-RV_001_161.jpeg: 960x736 1 Good, 324.0ms
image 4/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\045c881d-RV_001_63.jpeg: 960x736 1 Good, 1 Weathered, 386.9ms
image 5/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\055b0df8-IMG_4576.jpeg: 960x736 1 Good, 1 Weathered, 272.3ms
image 6/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\05a0d473-IMG20260105131437.jpg: 960x736 1 Good, 367.1ms
image 7/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\0b36e2e7-IMG_4686.jpeg: 960x736 1 Good, 235.9ms
image 8/110 C:\ARI3129_work\Datasets\YOLO_COCO_condition\images\test\0d36136c-IMG_4581.jpeg: 960x736 1 Good, 272.9ms
image 9/110

## More evaluation details

In [10]:
CONDITIONS = ["Good", "Weathered", "Heavily damaged"]

def extract_condition_from_name(name: str) -> str:
    n = name.lower()
    if "good" in n:
        return "Good"
    if "weathered" in n:
        return "Weathered"
    if "heavily" in n or "heavy" in n or "damaged" in n:
        return "Heavily damaged"
    return "Unknown"


import json
from collections import Counter

def export_ultralytics_condition_analytics(
    results,
    out_json="test_condition_analytics.json",
    out_csv="test_condition_analytics.csv",
):
    out_json = Path(out_json)
    out_csv = Path(out_csv)

    # auto create output directories if needed
    if out_json.parent != Path("."):
        out_json.parent.mkdir(parents=True, exist_ok=True)
    if out_csv.parent != Path("."):
        out_csv.parent.mkdir(parents=True, exist_ok=True)

    names = results[0].names if hasattr(results[0], "names") else None
    if names is None:
        raise ValueError("Could not find class names in results (results[0].names missing).")

    per_image = []
    total_condition_counts = Counter({c: 0 for c in CONDITIONS})
    total_detections = 0

    for r in results:
        filename = Path(r.path).name

        if r.boxes is None or len(r.boxes) == 0:
            cond_counts = {c: 0 for c in CONDITIONS}
            per_image.append({
                "filename": filename,
                "total_detections": 0,
                "condition_counts": cond_counts
            })
            continue

        cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        total_det = int(len(cls_ids))
        total_detections += total_det

        cond_counts = {c: 0 for c in CONDITIONS}
        for cid in cls_ids:
            cls_name = names[int(cid)]
            cond = extract_condition_from_name(cls_name)
            if cond in cond_counts:
                cond_counts[cond] += 1

        total_condition_counts.update(cond_counts)

        per_image.append({
            "filename": filename,
            "total_detections": total_det,
            "condition_counts": cond_counts
        })

    # JSON (per-image)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(per_image, f, indent=2)

    # CSV (flattened)
    rows = []
    for rec in per_image:
        row = {
            "filename": rec["filename"],
            "total_detections": rec["total_detections"],
        }
        for c in CONDITIONS:
            row[f"condition_{c}"] = rec["condition_counts"].get(c, 0)
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    return {
        "num_images": len(per_image),
        "total_detections": total_detections,
        "total_condition_counts": dict(total_condition_counts),
        "json_file": str(out_json.resolve()),
        "csv_file": str(out_csv.resolve()),
    }


summary = export_ultralytics_condition_analytics(
    results,
    out_json="2b_results/test_condition_analytics.json",
    out_csv="2b_results/test_condition_analytics.csv"
)
print(summary)



{'num_images': 110, 'total_detections': 138, 'total_condition_counts': {'Good': 116, 'Weathered': 22, 'Heavily damaged': 0}, 'json_file': 'C:\\Users\\User\\Desktop\\CV Assignment\\ARI3129 - Assignment Materials\\2b_results\\test_condition_analytics.json', 'csv_file': 'C:\\Users\\User\\Desktop\\CV Assignment\\ARI3129 - Assignment Materials\\2b_results\\test_condition_analytics.csv'}


In [11]:
df = pd.read_csv("2b_results/test_condition_analytics.csv")


condition_cols = [c for c in df.columns if c.startswith("condition_")]

condition_distribution = (
    df[condition_cols]
    .sum()
    .rename(lambda x: x.replace("condition_", ""))
    .reset_index()
)

condition_distribution.columns = ["Sign Condition", "Count"]

print("Distribution of Detected Traffic Sign Conditions")
print(condition_distribution)



total_images = len(df)
total_detections = int(df["total_detections"].sum())
mean_detections = total_detections / total_images
max_detections = int(df["total_detections"].max())

stats_table = pd.DataFrame({
    "Metric": [
        "Total test images",
        "Total traffic signs detected",
        "Mean signs per image",
        "Maximum signs in a single image",
    ],
    "Value": [
        total_images,
        total_detections,
        round(mean_detections, 2),
        max_detections,
    ],
})

print("\nDetection Statistics")
print(stats_table)

print("\nmAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)


Distribution of Detected Traffic Sign Conditions
    Sign Condition  Count
0             Good    116
1        Weathered     22
2  Heavily damaged      0

Detection Statistics
                            Metric   Value
0                Total test images  110.00
1     Total traffic signs detected  138.00
2             Mean signs per image    1.25
3  Maximum signs in a single image    4.00

mAP50: 0.7305014765302023
mAP50-95: 0.6982774174576719
Precision: 0.6726190476190477
Recall: 0.7595238095238095
